## 1) Dependencias

In [1]:
# %pip install pandas python-dateutil
import os, json, subprocess
from pathlib import Path
from datetime import date, datetime

import pandas as pd

## 2) Parámetros

In [2]:
RAW_DAY   = date.today().isoformat()        
VENDORS   = ["sbs", "crisol"]

LOCAL_IN  = "/srv/bigdata/input_simulados"  # <- según indicaste
LOCAL_OUT = "/srv/bigdata"                  # <- según indicaste

os.makedirs(LOCAL_OUT, exist_ok=True)

print("RAW_DAY   :", RAW_DAY)
print("LOCAL_IN  :", LOCAL_IN)
print("LOCAL_OUT :", LOCAL_OUT)

RAW_DAY   : 2025-11-09
LOCAL_IN  : /srv/bigdata/input_simulados
LOCAL_OUT : /srv/bigdata


## 3) Utilidades

In [3]:
from dateutil.parser import parse as dtparse

def run(*args, check=True):
    return subprocess.run(list(args), check=check, text=True, capture_output=True)

def read_csv_req(path, **kw):
    assert os.path.exists(path), f"No existe: {path}"
    return pd.read_csv(path, **kw)

def as_str(s):
    return s.astype("string").fillna("")

def to_num(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def to_date(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.date
    return df

def drop_dups(df, subset):
    before = len(df)
    df = df.drop_duplicates(subset=subset, keep="first")
    return df, before - len(df)

def pct(x, y):
    return 0.0 if y == 0 else round(100.0 * x / y, 2)

## 4) Esquemas

In [8]:
USERS_EXPECTED_SCHEMAS = {
    "sbs":    ["usuario_id", "edad", "genero", "region", "tipo_usuario", "fecha_registro"],
    "crisol": ["usuario_id", "nombre", "apellido", "edad", "genero", "region", "tipo_usuario", "fecha_registro"]
}

TX_EXPECTED = [
    "transaction_id", "fecha_compra", "usuario_id", "region", "n_items", 
    "total_transaction", "medio_pago", "estado_pago"
]

TX_ITEMS_EXPECTED = [
    "transaction_id", "line_id", "titulo", "cantidad", "precio_unitario", 
    "descuento_aplicado_pct", "precio_final_unitario", "line_total"
]

## 5) Carga + QA + Normalzación por vendor

In [9]:
summary = {}

for vendor in VENDORS:
    print("\n" + "="*90)
    print(f"==> VENDOR: {vendor.upper()} | ingest_date={RAW_DAY}")
    print("="*90)

    # 5.1 rutas de entrada
    f_users  = os.path.join(LOCAL_IN, f"usuarios_{vendor}.csv")
    f_tx     = os.path.join(LOCAL_IN, f"transacciones_{vendor}.csv")
    f_tx_itm = os.path.join(LOCAL_IN, f"transaccion_items_{vendor}.csv")

    # 5.2 leer archivos
    users  = read_csv_req(f_users)
    tx     = read_csv_req(f_tx)
    tx_itm = read_csv_req(f_tx_itm)

    # 5.3 validar columnas
    expected_users = USERS_EXPECTED_SCHEMAS[vendor]

    miss_u = [c for c in expected_users if c not in users.columns]
    miss_t = [c for c in TX_EXPECTED if c not in tx.columns]
    miss_i = [c for c in TX_ITEMS_EXPECTED if c not in tx_itm.columns]

    assert not miss_u, f"[{vendor}] faltan columnas en usuarios: {miss_u}"
    assert not miss_t, f"[{vendor}] faltan columnas en transacciones: {miss_t}"
    assert not miss_i, f"[{vendor}] faltan columnas en transaccion_items: {miss_i}"

    # 5.4 tipado mínimo
    for c in ["usuario_id","genero","region","tipo_usuario","estado_pago","medio_pago","transaction_id"]:
        if c in users.columns: users[c] = as_str(users[c])
        if c in tx.columns:    tx[c]    = as_str(tx[c])
        if c in tx_itm.columns: tx_itm[c] = as_str(tx_itm[c])

    if "titulo" in tx_itm.columns:
        tx_itm["titulo"] = as_str(tx_itm["titulo"])

    users  = to_num(users, ["edad"])
    tx     = to_num(tx, ["n_items", "total_transaction"])
    tx_itm = to_num(tx_itm, ["cantidad","precio_unitario","descuento_aplicado_pct",
                             "precio_final_unitario","line_total"])

    users = to_date(users, ["fecha_registro"])
    tx    = to_date(tx,    ["fecha_compra"])

    # 5.5 eliminar duplicados
    users,   dups_u   = drop_dups(users, subset=["usuario_id"])
    tx,      dups_t   = drop_dups(tx,    subset=["transaction_id"])
    tx_itm,  dups_itm = drop_dups(tx_itm, subset=["transaction_id", "line_id"])

    # 5.6 FK usuarios↔transacciones
    fk_ok = tx["usuario_id"].isin(users["usuario_id"]).sum()
    fk_total = len(tx)
    fk_pct = pct(fk_ok, fk_total)

    # 5.7 añadir vendor e ingest_date
    for df in [users, tx, tx_itm]:
        df["vendor"] = vendor
        df["ingest_date"] = RAW_DAY

    # 5.8 guardar limpios locales
    out_users     = os.path.join(LOCAL_OUT, f"{vendor}_usuarios_clean.csv")
    out_tx        = os.path.join(LOCAL_OUT, f"{vendor}_transacciones_clean.csv")
    out_tx_items  = os.path.join(LOCAL_OUT, f"{vendor}_transaccion_items_clean.csv")

    users.to_csv(out_users, index=False, encoding="utf-8-sig")
    tx.to_csv(out_tx,       index=False, encoding="utf-8-sig")
    tx_itm.to_csv(out_tx_items, index=False, encoding="utf-8-sig")

    print(f"[{vendor}] usuarios       -> {users.shape} (dups: {dups_u})")
    print(f"[{vendor}] transacciones  -> {tx.shape} (dups: {dups_t})")
    print(f"[{vendor}] items tx       -> {tx_itm.shape} (dups: {dups_itm})")
    print(f"[{vendor}] FK cobertura   : {fk_ok}/{fk_total} = {fk_pct}%")

    # 5.9 resumen por vendor
    summary[vendor] = {
        "users_path": out_users,
        "tx_path": out_tx,
        "tx_items_path": out_tx_items,
        "users_shape": list(users.shape),
        "tx_shape": list(tx.shape),
        "tx_items_shape": list(tx_itm.shape),
        "dups_removed": {"users": int(dups_u), "tx": int(dups_t), "tx_items": int(dups_itm)},
        "fk_coverage_pct": fk_pct,
        "dates": {
            "users": {"min": str(users["fecha_registro"].min()), "max": str(users["fecha_registro"].max())},
            "tx":    {"min": str(tx["fecha_compra"].min()), "max": str(tx["fecha_compra"].max())}
        }
    }

print("\n[OK] Limpieza y validación completadas.")


==> VENDOR: SBS | ingest_date=2025-11-09
[sbs] usuarios       -> (4358, 8) (dups: 1)
[sbs] transacciones  -> (39780, 10) (dups: 0)
[sbs] items tx       -> (99188, 10) (dups: 0)
[sbs] FK cobertura   : 39780/39780 = 100.0%

==> VENDOR: CRISOL | ingest_date=2025-11-09
[crisol] usuarios       -> (7500, 10) (dups: 0)
[crisol] transacciones  -> (67560, 10) (dups: 0)
[crisol] items tx       -> (168380, 10) (dups: 0)
[crisol] FK cobertura   : 67560/67560 = 100.0%

[OK] Limpieza y validación completadas.


## 6) Manifiestos

In [10]:
def sha256(path):
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

for vendor, info in summary.items():
    manifest = {
        "version": "1.0",
        "ingest_date": RAW_DAY,
        "vendor": vendor,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "files": {
            "users_csv": {
                "path": info["users_path"],
                "sha256": sha256(info["users_path"])
            },
            "tx_csv": {
                "path": info["tx_path"],
                "sha256": sha256(info["tx_path"])
            },
            "tx_items_csv": {
                "path": info["tx_items_path"],
                "sha256": sha256(info["tx_items_path"])
            }
        },
        "shapes": {
            "users": info["users_shape"],
            "tx":    info["tx_shape"],
            "tx_items": info["tx_items_shape"]
        },
        "dups_removed": info["dups_removed"],
        "fk_coverage_pct": info["fk_coverage_pct"],
        "dates": info["dates"]
    }

    out = os.path.join(LOCAL_OUT, f"{vendor}_ventas_manifest.json")
    with open(out, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    summary[vendor]["manifest_path"] = out
    print(f"[{vendor}] manifest -> {out}")

[sbs] manifest -> /srv/bigdata/sbs_ventas_manifest.json
[crisol] manifest -> /srv/bigdata/crisol_ventas_manifest.json


## 7) Subir a HDFS

In [11]:
for vendor, info in summary.items():
    HDFS_DIR = f"/data/raw/ventas/{vendor}/ingest_date={RAW_DAY}/"
    print(f"\n[HDFS] Subiendo {vendor} -> {HDFS_DIR}")
    run("hdfs", "dfs", "-mkdir", "-p", HDFS_DIR)

    run("hdfs", "dfs", "-put", "-f", info["users_path"], HDFS_DIR)
    run("hdfs", "dfs", "-put", "-f", info["tx_path"], HDFS_DIR)
    run("hdfs", "dfs", "-put", "-f", info["tx_items_path"], HDFS_DIR)
    run("hdfs", "dfs", "-put", "-f", info["manifest_path"], HDFS_DIR)

    run("hdfs", "dfs", "-touchz", HDFS_DIR + "_SUCCESS", check=False)

    ls_out = run("hdfs", "dfs", "-ls", "-h", HDFS_DIR, check=False)
    print(ls_out.stdout or ls_out.stderr)

print("\n[OK] Subida HDFS completada.")


[HDFS] Subiendo sbs -> /data/raw/ventas/sbs/ingest_date=2025-11-09/
Found 5 items
-rw-rw-r--+  2 bigdata bd          0 2025-11-09 16:20 /data/raw/ventas/sbs/ingest_date=2025-11-09/_SUCCESS
-rw-rw-r--+  2 bigdata bd      9.9 M 2025-11-09 16:20 /data/raw/ventas/sbs/ingest_date=2025-11-09/sbs_transaccion_items_clean.csv
-rw-rw-r--+  2 bigdata bd      3.3 M 2025-11-09 16:20 /data/raw/ventas/sbs/ingest_date=2025-11-09/sbs_transacciones_clean.csv
-rw-rw-r--+  2 bigdata bd    258.2 K 2025-11-09 16:20 /data/raw/ventas/sbs/ingest_date=2025-11-09/sbs_usuarios_clean.csv
-rw-rw-r--+  2 bigdata bd      1.0 K 2025-11-09 16:20 /data/raw/ventas/sbs/ingest_date=2025-11-09/sbs_ventas_manifest.json


[HDFS] Subiendo crisol -> /data/raw/ventas/crisol/ingest_date=2025-11-09/
Found 5 items
-rw-rw-r--+  2 bigdata bd          0 2025-11-09 16:21 /data/raw/ventas/crisol/ingest_date=2025-11-09/_SUCCESS
-rw-rw-r--+  2 bigdata bd     16.7 M 2025-11-09 16:21 /data/raw/ventas/crisol/ingest_date=2025-11-09/crisol_tr

## 8) Verificaciones en HDFS

In [12]:
for vendor in VENDORS:
    HDFS_DIR = f"/data/raw/ventas/{vendor}/ingest_date={RAW_DAY}/"
    print(f"\n[HDFS QA] {vendor} -> {HDFS_DIR}")

    cnt = run("hdfs", "dfs", "-count", "-h", HDFS_DIR)
    du  = run("hdfs", "dfs", "-du", "-h", HDFS_DIR)
    fsck = run("hdfs", "fsck", HDFS_DIR, "-files", "-blocks", "-racks")

    print(cnt.stdout or cnt.stderr)
    print(du.stdout or du.stderr)
    for line in fsck.stdout.splitlines():
        if any(k in line for k in ["Over-replicated","Under replicated","replicas"]):
            print(" ", line)

print("\nProceso de ingesta (VM2) completado.")


[HDFS QA] sbs -> /data/raw/ventas/sbs/ingest_date=2025-11-09/
           1            5             13.4 M /data/raw/ventas/sbs/ingest_date=2025-11-09

0        0        /data/raw/ventas/sbs/ingest_date=2025-11-09/_SUCCESS
9.9 M    19.8 M   /data/raw/ventas/sbs/ingest_date=2025-11-09/sbs_transaccion_items_clean.csv
3.3 M    6.6 M    /data/raw/ventas/sbs/ingest_date=2025-11-09/sbs_transacciones_clean.csv
258.2 K  516.5 K  /data/raw/ventas/sbs/ingest_date=2025-11-09/sbs_usuarios_clean.csv
1.0 K    2.0 K    /data/raw/ventas/sbs/ingest_date=2025-11-09/sbs_ventas_manifest.json

   Over-replicated blocks:	0 (0.0 %)
   Missing replicas:		0 (0.0 %)

[HDFS QA] crisol -> /data/raw/ventas/crisol/ingest_date=2025-11-09/
           1            5             23.1 M /data/raw/ventas/crisol/ingest_date=2025-11-09

0        0       /data/raw/ventas/crisol/ingest_date=2025-11-09/_SUCCESS
16.7 M   33.4 M  /data/raw/ventas/crisol/ingest_date=2025-11-09/crisol_transaccion_items_clean.csv
5.8 M    11.7 M 